# Random Forest
Εκπαιδεύουμε πολλά variations του μοντέλου με διαφορετικές υπερπαραμέτρους και βρίσκουμε το μοντέλο με τις καλύτερες υπερπαραμέτρους. Ύστερα, το εφαρμόζουμε πάνω στο test gold για να κάνουμε προβλέψεις και τις αποθηκεύουμε για να γίνουν evaluate μαζί με όλες τις άλλες 

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print("1. Εκκίνηση SparkSession...")
spark = SparkSession.builder \
    .appName("RF_SMOTE_Corrected_GridSearch") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("2. Φόρτωση και Feature Augmentation...")
train_gold = spark.read.parquet("../../data/train_gold_with_cluster.parquet")
test_gold = spark.read.parquet("../../data/test_gold_with_cluster.parquet")

train_gold = train_gold.withColumn("stroke", train_gold["stroke"].cast(DoubleType()))
test_gold = test_gold.withColumn("stroke", test_gold["stroke"].cast(DoubleType()))
train_gold = train_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))
test_gold = test_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))

assembler = VectorAssembler(inputCols=["features", "cluster"], outputCol="augmented_features")
train_augmented = assembler.transform(train_gold).drop("features").withColumnRenamed("augmented_features", "features")
test_augmented = assembler.transform(test_gold).drop("features").withColumnRenamed("augmented_features", "features")

# Κρατάμε ολόκληρο το SMOTE dataset (χωρίς undersampling)
train_augmented.cache()

print("3. Ορισμός Πλέγματος Παραμέτρων (Grid Search)...")
rf_base = RandomForestClassifier(featuresCol="features", labelCol="stroke", seed=22390225)

# ΔΙΚΑΙΟΛΟΓΗΣΗ (Η ΔΙΟΡΘΩΣΗ): 
# Κατεβάζουμε το maxDepth στα 4, 6 και 8. Αυτό εμποδίζει το δέντρο να 
# "αποστηθίσει" τα συνθετικά δεδομένα του SMOTE, κρατώντας το Recall ψηλά 
# για τα πραγματικά, άγνωστα δεδομένα του Test Set.
paramGrid = (ParamGridBuilder()
             .addGrid(rf_base.maxDepth, [4, 6, 8]) 
             .addGrid(rf_base.numTrees, [100, 200])
             .build())

# Χρήση PR-AUC για την αξιολόγηση 
evaluator = BinaryClassificationEvaluator(labelCol="stroke", rawPredictionCol="rawPrediction", metricName="areaUnderPR")

cv = CrossValidator(estimator=rf_base,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=3,
                    seed=22390225)

print("4. Εκτέλεση Cross-Validation στο ΠΛΗΡΕΣ SMOTE Dataset...")
cv_model = cv.fit(train_augmented)
best_rf = cv_model.bestModel

print("\n===========================================================")
print("[ΔΙΟΡΘΩΜΕΝΗ ΕΥΡΕΣΗ SMOTE] Οι βέλτιστες παράμετροι βρέθηκαν:")
print(f" -> maxDepth: {best_rf._java_obj.getMaxDepth()}")
print(f" -> numTrees: {best_rf._java_obj.getNumTrees()}")
print("===========================================================")

print("5. Παραγωγή και αποθήκευση προβλέψεων στο Test Set...")
rf_preds = best_rf.transform(test_augmented)

# Αποθήκευση ως το κεντρικό rf_predictions
rf_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../../data/rf_predictions.parquet")

spark.stop()
print("Ολοκληρώθηκε! Έχεις πλέον ένα RF που αντιστέκεται στο overfitting του SMOTE.")

1. Εκκίνηση SparkSession...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/11 16:27:41 WARN Utils: Your hostname, cachyos-x8664, resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface enp4s0)
26/06/11 16:27:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 16:27:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/11 16:27:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


2. Φόρτωση και Feature Augmentation...
3. Ορισμός Πλέγματος Παραμέτρων (Grid Search)...
4. Εκτέλεση Cross-Validation στο ΠΛΗΡΕΣ SMOTE Dataset...


26/06/11 16:27:51 WARN DAGScheduler: Broadcasting large task binary with size 1652.9 KiB
26/06/11 16:27:52 WARN DAGScheduler: Broadcasting large task binary with size 1492.3 KiB
26/06/11 16:27:53 WARN DAGScheduler: Broadcasting large task binary with size 1367.9 KiB
26/06/11 16:27:53 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/06/11 16:27:53 WARN DAGScheduler: Broadcasting large task binary with size 1555.4 KiB
26/06/11 16:27:54 WARN DAGScheduler: Broadcasting large task binary with size 1652.9 KiB
26/06/11 16:27:54 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/11 16:27:55 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
26/06/11 16:27:56 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
26/06/11 16:27:59 WARN DAGScheduler: Broadcasting large task binary with size 1656.9 KiB
26/06/11 16:28:00 WARN DAGScheduler: Broadcasting large task binary with size 1523.8 KiB
26/06/11 16:28:01 WARN DAGSchedul


[ΔΙΟΡΘΩΜΕΝΗ ΕΥΡΕΣΗ SMOTE] Οι βέλτιστες παράμετροι βρέθηκαν:
 -> maxDepth: 8
 -> numTrees: 200
5. Παραγωγή και αποθήκευση προβλέψεων στο Test Set...


26/06/11 16:28:14 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB


Ολοκληρώθηκε! Έχεις πλέον ένα RF που αντιστέκεται στο overfitting του SMOTE.
